# E791-style fit with efficiency and background

The $D^+\to\pi^-\pi^+\pi^+$ model uses mass-plane Gauss--Legendre normalization. The efficiency and background are configured once; `generate_toy` uses them for pseudo-data generation and `FitSession` reuses the same objects for the fit and normalization. Toy generation uses the default inverse-transform sampler.


In [ ]:
import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt

from dalitzplotfitter import (
    BackgroundSpec, DecayChannel, DecayModel, NonResonant, Parameter, RealImag, Resonance,
    FitSession, ToyBackground, enable_x64, generate_toy, plot_dalitz,
)
from dalitzplotfitter.background import FunctionalBackground
from dalitzplotfitter.efficiency import FunctionalEfficiency

enable_x64()


In [ ]:
channel = DecayChannel("D+", ("pi-", "pi+", "pi+"))
nr_x = Parameter.coefficient("NR.x", 0.45, bounds=(-2, 2), step=0.02, owner="NR")
nr_y = Parameter.coefficient("NR.y", -0.25, bounds=(-2, 2), step=0.02, owner="NR")
model = DecayModel(
    channel,
    [
        Resonance("rho", (0, 1), RealImag(1.0, 0.0),
                  mass=0.7753, width=0.1491, spin=1),
        NonResonant(RealImag(nr_x, nr_y)),
    ],
    normalization_method="gauss-legendre",
    normalization_order_m13=180,
    normalization_order_m23=180,
)
truth = {"NR.x": 0.45, "NR.y": -0.25}

efficiency = FunctionalEfficiency(
    lambda d: 0.55 + 0.30*jnp.clip((d["s13"]-0.1)/2.5, 0, 1)
)
background = FunctionalBackground(
    lambda d: 0.3 + 0.7*jnp.clip((d["s23"]-0.1)/2.5, 0, 1)
)
f_sig = Parameter("signal_fraction", 0.75, bounds=(0.05, 0.99), step=0.01)


In [ ]:
data = generate_toy(
    model, 25_000, parameters=truth, efficiency=efficiency,
    signal_fraction=0.78,
    backgrounds=(ToyBackground("combinatorial", background),),
    seed=202,
)
plot_dalitz(data, x="s13", y="s23", title="Efficiency + background toy")
plt.show()


In [ ]:
session = FitSession(
    model, data, efficiency=efficiency,
    signal_fraction=f_sig,
    backgrounds=(BackgroundSpec("combinatorial", background),),
)
result = session.fit({"NR.x": 0.15, "NR.y": 0.05, "signal_fraction": 0.68},
                     simplex=True, ncall=35_000)
session.report(result)
session.plot_projection(result, "s13")
plt.show()
